### Cell 1 -  Setup

In [0]:
# ============================================================
#  02_SILVER_CLEANING — TOURGO DATA PIPELINE
#  Mục đích: Bronze Managed Tables -> Cleaning -> Silver Managed Tables
# ============================================================

from pyspark.sql.functions import (
    col, to_date, to_timestamp, when, trim, abs as _abs
)
from pyspark.sql.types import DoubleType, IntegerType, BooleanType

# Dùng chung database tourgo đã tạo ở notebook 01
spark.sql("USE tourgo")
print("   Sử dụng Database: tourgo")
print("   Bronze tables: bronze_users, bronze_tours, ...")
print("   Silver tables sẽ tạo: silver_users, silver_tours, ...")

   Sử dụng Database: tourgo
   Bronze tables: bronze_users, bronze_tours, ...
   Silver tables sẽ tạo: silver_users, silver_tours, ...


### Cell 2 - Silver Users

In [0]:
# ── USERS ──────────────────────────────────────────────────
print("Cleaning USERS...")

df_u = spark.read.table("bronze_users")
bronze_count = df_u.count()

df_users_s = df_u \
    .dropDuplicates(["id"]) \
    .filter(col("role").isin(["ADMIN", "PROVIDER", "CUSTOMER"])) \
    .withColumn("is_active",   col("is_active").cast(BooleanType())) \
    .withColumn("date_joined", to_timestamp(col("date_joined")))

df_users_s.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver_users")

silver_count = df_users_s.count()
print(f"   Bronze: {bronze_count:,} → Silver: {silver_count:,} (dropped: {bronze_count - silver_count})")
df_users_s.show(3, truncate=False)

Cleaning USERS...
   Bronze: 18 → Silver: 0 (dropped: 18)
+---+----+---------+-----------+
|id |role|is_active|date_joined|
+---+----+---------+-----------+
+---+----+---------+-----------+



### Cell 3 - Silver Tours

In [0]:
# ── TOURS ───────────────────────────────────────────────────
print("Cleaning TOURS...")

df_t = spark.read.table("bronze_tours")
bronze_count = df_t.count()

df_tours_s = df_t \
    .dropDuplicates(["id"]) \
    .withColumn("price", col("price").cast(DoubleType())) \
    .withColumn("slots", col("slots").cast(IntegerType())) \
    .filter(col("price") > 0) \
    .filter(col("slots") >= 0) \
    .withColumn("departure_date", to_date(col("departure_date"))) \
    .withColumn("created_at",     to_timestamp(col("created_at"))) \
    .withColumn("category_names", trim(col("category_names")))

df_tours_s.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver_tours")

silver_count = df_tours_s.count()
print(f"   Bronze: {bronze_count:,} → Silver: {silver_count:,} (dropped: {bronze_count - silver_count})")
df_tours_s.show(3, truncate=False)

Cleaning TOURS...
   Bronze: 23 → Silver: 23 (dropped: 0)
+---+----------+------------------------------------+--------+--------------+-----+--------+-------------------+--------------+
|id |creator_id|title                               |price   |departure_date|slots|status  |created_at         |category_names|
+---+----------+------------------------------------+--------+--------------+-----+--------+-------------------+--------------+
|8  |13        |Tour săn mây và đón bình minh Đà Lạt|500000.0|2026-05-24    |12   |pending |2026-05-23 11:19:57|NULL          |
|9  |13        |Delight Park Đà Lạt                 |150000.0|2026-05-02    |1    |approved|2026-05-23 16:15:00|NULL          |
|10 |11        |Lotte World Aquarium                |180000.0|2026-05-31    |0    |approved|2026-05-23 16:21:45|NULL          |
+---+----------+------------------------------------+--------+--------------+-----+--------+-------------------+--------------+
only showing top 3 rows


### Cell 4 - Silver Bookings

In [0]:
# ── BOOKINGS ────────────────────────────────────────────────
print("Cleaning BOOKINGS...")

df_b = spark.read.table("bronze_bookings")
bronze_count = df_b.count()

df_bookings_s = df_b \
    .dropDuplicates(["id"]) \
    .filter(col("status").isin(["pending", "confirmed", "cancelled"])) \
    .withColumn("total_price",      col("total_price").cast(DoubleType())) \
    .withColumn("number_of_people", col("number_of_people").cast(IntegerType())) \
    .withColumn("booking_date",     to_date(col("booking_date"))) \
    .withColumn("created_at",       to_timestamp(col("created_at")))

df_bookings_s.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver_bookings")

silver_count = df_bookings_s.count()
print(f"   Bronze: {bronze_count:,} → Silver: {silver_count:,} (dropped: {bronze_count - silver_count})")
df_bookings_s.show(5, truncate=False)

Cleaning BOOKINGS...
   Bronze: 11 → Silver: 11 (dropped: 0)
+---+-------+-------+----------------+-----------+------------+---------+-------------------+
|id |user_id|tour_id|number_of_people|total_price|booking_date|status   |created_at         |
+---+-------+-------+----------------+-----------+------------+---------+-------------------+
|6  |10     |10     |1               |180000.0   |2026-05-31  |confirmed|2026-05-24 03:57:32|
|7  |10     |11     |1               |70000.0    |2026-05-30  |cancelled|2026-05-24 04:10:03|
|8  |10     |10     |1               |180000.0   |2026-05-31  |cancelled|2026-05-24 04:31:26|
|9  |10     |10     |2               |180000.0   |2026-05-31  |confirmed|2026-05-24 04:34:54|
|10 |10     |12     |1               |4000000.0  |2026-06-24  |confirmed|2026-05-24 04:41:15|
+---+-------+-------+----------------+-----------+------------+---------+-------------------+
only showing top 5 rows


### Cell 5 - Silver Payments

In [0]:
# ── PAYMENTS ────────────────────────────────────────────────
print("Cleaning PAYMENTS...")

df_p = spark.read.table("bronze_payments")
bronze_count = df_p.count()

df_payments_s = df_p \
    .dropDuplicates(["id"]) \
    .filter(col("payment_method").isin(["VietQR", "VNPay"])) \
    .withColumn("amount",     col("amount").cast(DoubleType())) \
    .withColumn("created_at", to_timestamp(col("created_at")))

df_payments_s.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver_payments")

silver_count = df_payments_s.count()
print(f"   Bronze: {bronze_count:,} → Silver: {silver_count:,} (dropped: {bronze_count - silver_count})")
df_payments_s.show(5, truncate=False)

Cleaning PAYMENTS...
   Bronze: 10 → Silver: 10 (dropped: 0)
+---+----------+---------+--------------+--------------+-------------------+
|id |booking_id|amount   |payment_method|status        |created_at         |
+---+----------+---------+--------------+--------------+-------------------+
|1  |6         |180000.0 |VietQR        |AWAITING_ADMIN|2026-05-24 03:57:36|
|2  |7         |70000.0  |VietQR        |AWAITING_ADMIN|2026-05-24 04:10:08|
|3  |8         |180000.0 |VietQR        |AWAITING_ADMIN|2026-05-24 04:31:31|
|4  |9         |180000.0 |VietQR        |AWAITING_ADMIN|2026-05-24 04:35:01|
|5  |10        |4000000.0|VietQR        |SUCCESS       |2026-05-24 04:41:19|
+---+----------+---------+--------------+--------------+-------------------+
only showing top 5 rows


### Cell 6 - Silver Revenues

In [0]:
# ── REVENUES ────────────────────────────────────────────────
# Dùng tolerance 0.01 thay vì == để tránh floating point error
print("Cleaning REVENUES...")

df_r = spark.read.table("bronze_revenues")
bronze_count = df_r.count()

df_revenues_s = df_r \
    .dropDuplicates(["id"]) \
    .withColumn("total_amount",  col("total_amount").cast(DoubleType())) \
    .withColumn("creator_share", col("creator_share").cast(DoubleType())) \
    .withColumn("admin_share",   col("admin_share").cast(DoubleType())) \
    .withColumn("created_at",    to_timestamp(col("created_at"))) \
    .filter(
        _abs(col("creator_share") + col("admin_share") - col("total_amount")) < 0.01
    )

df_revenues_s.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver_revenues")

silver_count = df_revenues_s.count()
print(f"   Bronze: {bronze_count:,} → Silver: {silver_count:,} (dropped: {bronze_count - silver_count})")
df_revenues_s.show(5, truncate=False)

Cleaning REVENUES...
   Bronze: 4 → Silver: 4 (dropped: 0)
+---+----------+----------+------------+-------------+-----------+-------------------+
|id |payment_id|creator_id|total_amount|creator_share|admin_share|created_at         |
+---+----------+----------+------------+-------------+-----------+-------------------+
|1  |5         |11        |4000000.0   |3600000.0    |400000.0   |2026-05-24 04:41:36|
|2  |8         |17        |70000.0     |63000.0      |7000.0     |2026-05-24 15:26:26|
|3  |7         |17        |70000.0     |63000.0      |7000.0     |2026-05-24 15:26:33|
|4  |10        |11        |1000000.0   |900000.0     |100000.0   |2026-05-29 06:51:21|
+---+----------+----------+------------+-------------+-----------+-------------------+



### Cell 7 - Silver Reviews

In [0]:
# ── REVIEWS ─────────────────────────────────────────────────
print("Cleaning REVIEWS...")

df_rv = spark.read.table("bronze_reviews")
bronze_count = df_rv.count()

df_reviews_s = df_rv \
    .dropDuplicates(["id"]) \
    .withColumn("rating", col("rating").cast(IntegerType())) \
    .filter(col("rating").isNotNull()) \
    .filter((col("rating") >= 1) & (col("rating") <= 5)) \
    .withColumn("created_at", to_timestamp(col("created_at")))

df_reviews_s.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver_reviews")

silver_count = df_reviews_s.count()
print(f"   Bronze: {bronze_count:,} → Silver: {silver_count:,} (dropped: {bronze_count - silver_count})")
df_reviews_s.show(3, truncate=False)

Cleaning REVIEWS...
   Bronze: 1 → Silver: 1 (dropped: 0)
+---+-------+-------+------+-------------------+
|id |user_id|tour_id|rating|created_at         |
+---+-------+-------+------+-------------------+
|2  |10     |12     |5     |2026-05-24 15:29:02|
+---+-------+-------+------+-------------------+



### Cell 8 - Comparison Report (chụp screenshot)

In [0]:
# ============================================================
#  BRONZE vs SILVER COMPARISON — chụp screenshot cell này
# ============================================================

TABLES = ['users', 'tours', 'bookings', 'payments', 'revenues', 'reviews']

print("\n" + "="*55)
print("BRONZE vs SILVER COMPARISON")
print("="*55)
print(f"{'Table':<12} | {'Bronze':>8} | {'Silver':>8} | {'Dropped':>8}")
print("-"*45)

for t in TABLES:
    try:
        b = spark.read.table(f"bronze_{t}").count()
        s = spark.read.table(f"silver_{t}").count()
        flag = "--(ERROR)--" if (b - s) > 0 else "-OK-"
        print(f"{t:<12} | {b:>8,} | {s:>8,} | {b-s:>8,}{flag}")
    except Exception as e:
        print(f"{t:<12} | {'ERR':>8} | {'ERR':>8} | {str(e)[:25]}")

print("="*55)
print("Silver Layer hoàn tất — sẵn sàng cho Day 3 (Gold)")


BRONZE vs SILVER COMPARISON
Table        |   Bronze |   Silver |  Dropped
---------------------------------------------
users        |       18 |        0 |       18--(ERROR)--
tours        |       23 |       23 |        0-OK-
bookings     |       11 |       11 |        0-OK-
payments     |       10 |       10 |        0-OK-
revenues     |        4 |        4 |        0-OK-
reviews      |        1 |        1 |        0-OK-
Silver Layer hoàn tất — sẵn sàng cho Day 3 (Gold)


### Cell 10 - Gửi cho Khánh

In [0]:
# ============================================================
#  COPY KẾT QUẢ NÀY GỬI CHO [D] KHÁNH
#  Để Khang điền vào streaming_simulator.py


print("=" * 50)
print("DATA CHO [D] KHÁNH — streaming_simulator.py")
print("=" * 50)

df_users_info = spark.read.table("silver_users") \
    .filter(col("role") == "CUSTOMER") \
    .select("id") \
    .limit(10)

df_tours_info = spark.read.table("silver_tours") \
    .select("id", "price") \
    .limit(10)

user_ids = [row["id"] for row in df_users_info.collect()]
tour_rows = df_tours_info.collect()
tour_ids  = [row["id"]    for row in tour_rows]
prices    = list(set([int(row["price"]) for row in tour_rows]))

print(f"SAMPLE_USER_IDS = {user_ids}")
print(f"SAMPLE_TOUR_IDS = {tour_ids}")
print(f"SAMPLE_PRICES   = {prices}")
print("\n-> Copy 3 dòng trên paste vào streaming_simulator.py thay cho placeholder")

DATA CHO [D] KHÁNH — streaming_simulator.py
SAMPLE_USER_IDS = []
SAMPLE_TOUR_IDS = [28, 29, 30, 9, 11, 14, 17, 19, 23, 24]
SAMPLE_PRICES   = [1000000, 2000000, 1500000, 10000000, 70000, 150000, 1231123, 99999]

-> Copy 3 dòng trên paste vào streaming_simulator.py thay cho placeholder
